# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.qs2f-h81p) dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
%pip install -q mlcroissant

## 1. Data Loading

In this section, we'll load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's review the available **record sets** in the dataset, and for each record set, enumerate its **fields** (variables) and their Croissant `@id`s. This overview will help us select which table(s) to analyze.

In [ ]:
# List all record sets, their @id, and their fields
record_sets = list(dataset.record_sets())
if not record_sets:
    print("No record sets found directly in the schema. Listing all available via metadata...")
    # fallback for some schemas: peek at metadata.record_sets
    for obj in dir(metadata):
        if not obj.startswith('_'):
            value = getattr(metadata, obj)
            if hasattr(value, 'fields'):
                record_sets.append(value)
if not record_sets:
    raise RuntimeError("No record sets detected in this dataset.")

print("Record sets in this dataset:")
for rs in record_sets:
    print(f"- Record set name: {rs.name}\n  @id: {rs.id}")
    field_ids = []
    # List all fields in this record set
    for field in rs.fields:
        print(f"    - Field: {field.name} (@id: {field.id}, dataType: {field.data_type})")
        field_ids.append(field.id)
    print()

## 3. Data Extraction

We'll load data from all record sets into Pandas DataFrames for further analysis. Replace the example `@id`s below with those listed above as needed.

In [ ]:
# Extract data from all available record sets
dataframes = {}
for rs in record_sets:
    print(f"Loading records for record set: {rs.name} (@id: {rs.id})")
    recs = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(recs)
    dataframes[rs.id] = df
    print(f"  DataFrame columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}\n")
# For demonstration, pick the first record set for further EDA
main_record_set_id = record_sets[0].id
df = dataframes[main_record_set_id]
print(f"Using record set: {main_record_set_id}")
print(df.head())

## 4. Exploratory Data Analysis (EDA)

Let's 
- filter records based on a numeric attribute (e.g., Age),
- normalize the column,
- and group by a relevant categorical field (e.g., Sex or Cancer Location).

**All variables and DataFrame columns are referenced by their Croissant `@id` (not just the friendly name).**

In [ ]:
### Identify a numeric field and a categorical field by their @id (from data overview above)
# For this dataset, suppose 'schema:age' is the @id for Age, and 'schema:sex' for Sex
# Please adjust based on output from section 2 if needed.
# Fallbacks for demonstration:

# Helper to find likely numeric and categorical columns
numeric_field_id = None
categorical_field_id = None
for col in df.columns:
    if ("age" in col.lower()) and (numeric_field_id is None):
        numeric_field_id = col
    if ("sex" in col.lower() or "gender" in col.lower()) and (categorical_field_id is None):
        categorical_field_id = col
if numeric_field_id is None:
    # fallback: pick first float/int column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if categorical_field_id is None:
    # fallback: pick first object/str column
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):
            categorical_field_id = col
            break
print(f"Numeric field (by @id): {numeric_field_id}")
print(f"Categorical/grouping field (by @id): {categorical_field_id}")

# Remove records with null or non-numeric Age
df_sub = df.copy()
df_sub = df_sub[pd.to_numeric(df_sub[numeric_field_id], errors='coerce').notnull()]  # valid numeric values only
df_sub[numeric_field_id] = pd.to_numeric(df_sub[numeric_field_id], errors='coerce')

# Filter for Age > 50 (example threshold); adjust to domain
threshold = 50
filtered_df = df_sub[df_sub[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize Age field for filtered records
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by Sex (or other grouping field) and show average Age
if categorical_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(categorical_field_id)[numeric_field_id].mean().to_frame("mean_age")
    print(f"Grouped data by {categorical_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Visualize the age distribution and its breakdown by sex (or your chosen grouping variable), referencing fields by their `@id` as columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the normalized age field
plt.figure(figsize=(6,4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=15, kde=True)
plt.title(f"Distribution of normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id} (normalized)")
plt.ylabel("Count")
plt.show()

# Boxplot of age by group (sex)
if categorical_field_id in filtered_df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=filtered_df[categorical_field_id], y=filtered_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {categorical_field_id}")
    plt.xlabel(categorical_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, you have:

- Loaded a FAIR-compliant clinical record dataset using `mlcroissant` from its Croissant schema.
- Explored its structure, listing all record sets and their fields by `@id`.
- Extracted structured tabular data and performed exploratory analysis using field IDs.
- Filtered, normalized, grouped, and visualized key numeric and categorical fields—all while referencing data fields via their stable `@id`s, as recommended for FAIR processing.

For your own analysis, you can adapt this workflow to other Croissant datasets—always referencing fields and record sets by their explicit `@id` for robust, reproducible code.